# Runtime and GPU check

Select your Colab kernel in VS Code, then run all cells. The output describes the selected kernel's machine. A successful GPU check lists an NVIDIA GPU and, when PyTorch is installed with CUDA support, completes a small calculation on it.

In [1]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('System:', platform.platform())
print('Working directory:', Path.cwd())
print('Kernel connection works.')

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
System: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content
Kernel connection works.


In [2]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
        capture_output=True, text=True, timeout=30,
    )
    print(result.stdout or result.stderr)
    if result.returncode:
        print('GPU inspection failed; check the selected runtime.')
else:
    print('nvidia-smi is unavailable. Check that you selected a GPU runtime.')

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    if exc.name != 'torch':
        raise
    print('PyTorch is not installed in this kernel; skipping the tensor check.')
else:
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        values = torch.tensor([1.0, 2.0, 3.0], device='cuda')
        result = (values * values).sum().item()
        assert result == 14.0, f'Unexpected result: {result}'
        print('GPU calculation passed:', result)
    else:
        print('GPU calculation skipped. Check the runtime and PyTorch CUDA support.')

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU calculation passed: 14.0


In [ ]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

: 

In [1]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

No Hugging Face Dataset or DatasetDict found in the notebook.
Tell me what variable you used to load SLAKE, or paste your data-loading cell.


In [2]:
%pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
import json

train_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="train.json",
    repo_type="dataset",
)

with open(train_path, "r", encoding="utf-8") as file:
    slake_train_data = json.load(file)

print("Loaded file:", train_path)
print("Top-level type:", type(slake_train_data).__name__)

if isinstance(slake_train_data, list):
    print("Number of training examples:", len(slake_train_data))
    sample = slake_train_data[0]

elif isinstance(slake_train_data, dict):
    print("Top-level keys:", list(slake_train_data.keys()))

    # Look for a list of examples inside the JSON object
    rows = next(
        (
            value for value in slake_train_data.values()
            if isinstance(value, list) and value
        ),
        None,
    )

    if rows is not None:
        print("Number of examples in first list:", len(rows))
        sample = rows[0]
    else:
        sample = slake_train_data

else:
    sample = slake_train_data

print("\nFirst example:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:3000])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

Loaded file: /root/.cache/huggingface/hub/datasets--BoKelvin--SLAKE/snapshots/a9083ce6c34ac3ffb17671a605962924d8a8f9e9/train.json
Top-level type: list
Number of training examples: 9835

First example:
{
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "q_lang": "en",
  "location": "Abdomen",
  "modality": "MRI",
  "answer_type": "OPEN",
  "base_type": "vqa",
  "content_type": "Modality",
  "triple": [
    "vhead",
    "_",
    "_"
  ],
  "qid": 0
}


In [3]:
from huggingface_hub import hf_hub_download
from zipfile import ZipFile
import json

image_folder = slake_train_data[0]["img_name"].split("/")[0]
print("Image folder:", image_folder)

imgs_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset",
)

with ZipFile(imgs_zip_path) as archive:
    matching_files = []

    for path in archive.namelist():
        normalized_path = path.replace("\\", "/").strip("/")
        parts = normalized_path.split("/")

        if image_folder in parts:
            matching_files.append(normalized_path)

    print("\nFiles for this image:")
    for path in matching_files:
        print(" ", path)

    detection_files = [
        path for path in matching_files
        if path.lower().endswith("/detection.json")
        or path.lower() == "detection.json"
    ]

    if detection_files:
        detection_data = json.loads(archive.read(detection_files[0]))
        print("\nDetection annotation:")
        print(json.dumps(detection_data, indent=2, ensure_ascii=False)[:3000])
    else:
        print("\nNo detection.json found for this image folder.")

Image folder: xmlab1


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            


Files for this image:
  imgs/xmlab1
  imgs/xmlab1/question.json
  imgs/xmlab1/source.jpg
  imgs/xmlab1/mask.png
  imgs/xmlab1/detection.json

Detection annotation:
[
  {
    "Liver": [
      54.0,
      106.0,
      30.0,
      31.0
    ]
  }
]


In [4]:
from zipfile import ZipFile
from io import BytesIO
from PIL import Image

first_row = slake_train_data[0]
image_id = first_row["img_id"]
image_name = first_row["img_name"]

print("Image ID:", image_id)
print("Image name:", image_name)

print("\nQuestions for this image:")
for row in slake_train_data:
    if row.get("img_id") == image_id and row.get("q_lang") == "en":
        print(
            f"QID {row.get('qid')}: "
            f"{row.get('question')} "
            f"-> {row.get('answer')} "
            f"[{row.get('content_type')}]"
        )

source_file = next(
    (path for path in matching_files if path.lower().endswith("/source.jpg")),
    None,
)

if source_file:
    with ZipFile(imgs_zip_path) as archive:
        image = Image.open(BytesIO(archive.read(source_file)))
        print("\nImage dimensions (width, height):", image.size)
else:
    print("\nCould not find source.jpg in the matching files.")

Image ID: 1
Image name: xmlab1/source.jpg

Questions for this image:
QID 0: What modality is used to take this image? -> MRI [Modality]
QID 1: Which part of the body does this image belong to? -> Abdomen [Position]
QID 2: What is the mr weighting in this image? -> T2 [Modality]
QID 3: Does the picture contain liver? -> Yes [Organ]
QID 4: Does the picture contain kidney? -> No [Organ]
QID 5: Does the picture contain spleen? -> No [Organ]
QID 6: Where is the liver? -> Right [Position]

Image dimensions (width, height): (256, 256)


In [5]:
import json
from zipfile import ZipFile
from pathlib import PurePosixPath

image_name = "xmlab1/source.jpg"
image_folder = PurePosixPath(image_name).parent.as_posix()

# Find this image's detection.json inside the downloaded imgs.zip
with ZipFile(imgs_zip_path) as archive:
    detection_path = next(
        name for name in archive.namelist()
        if name.replace("\\", "/").endswith(
            f"{image_folder}/detection.json"
        )
    )
    detection_data = json.loads(archive.read(detection_path))

# Extract the Liver box from the annotation
liver_box = None
for item in detection_data:
    if "Liver" in item:
        liver_box = item["Liver"]
        break

# Make one label record per English question for this image
image_questions = [
    row for row in slake_train_data
    if row.get("img_name") == image_name
    and row.get("q_lang") == "en"
]

labels = []

for row in image_questions:
    question = row["question"].lower()
    answer = row["answer"].lower()

    asks_about_liver = "liver" in question
    has_positive_liver_answer = answer in {"yes", "right", "left"}

    use_liver_box = (
        asks_about_liver
        and has_positive_liver_answer
        and liver_box is not None
    )

    labels.append({
        "img_id": row["img_id"],
        "img_name": row["img_name"],
        "qid": row["qid"],
        "question": row["question"],
        "answer": row["answer"],
        "bbox_xywh_pixels": liver_box if use_liver_box else None,
        "grounding_status": (
            "grounded"
            if use_liver_box
            else "no_positive_region"
            if "does the picture contain" in question and answer == "no"
            else "not_required"
        )
    })

print(json.dumps(labels, indent=2))

[
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 0,
    "question": "What modality is used to take this image?",
    "answer": "MRI",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 1,
    "question": "Which part of the body does this image belong to?",
    "answer": "Abdomen",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 2,
    "question": "What is the mr weighting in this image?",
    "answer": "T2",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 3,
    "question": "Does the picture contain liver?",
    "answer": "Yes",
    "bbox_xywh_pixels": [
      54.0,
      106.0,
      30.0,
      31.0
    ],
    "grounding_status": "grounded"
  },
  {
    "img_id": 1,
    "img_name": "xm

In [6]:
for item in labels:
    print(
        f"QID {item['qid']}: answer={item['answer']}, "
        f"status={item['grounding_status']}, "
        f"box={item['bbox_xywh_pixels']}"
    )

QID 0: answer=MRI, status=not_required, box=None
QID 1: answer=Abdomen, status=not_required, box=None
QID 2: answer=T2, status=not_required, box=None
QID 3: answer=Yes, status=grounded, box=[54.0, 106.0, 30.0, 31.0]
QID 4: answer=No, status=no_positive_region, box=None
QID 5: answer=No, status=no_positive_region, box=None
QID 6: answer=Right, status=grounded, box=[54.0, 106.0, 30.0, 31.0]


In [7]:
from zipfile import ZipFile
from pathlib import PurePosixPath

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

with ZipFile(imgs_zip_path) as archive:
    files = [
        name.replace("\\", "/").lstrip("./")
        for name in archive.namelist()
    ]

    source_files = {
        name.removeprefix("imgs/")
        for name in files
        if name.endswith("/source.jpg")
    }

    detection_folders = {
        str(PurePosixPath(name).parent).removeprefix("imgs/")
        for name in files
        if name.endswith("/detection.json")
    }

question_images = {
    row["img_name"]
    for row in english_rows
}

question_folders = {
    PurePosixPath(name).parent.as_posix()
    for name in question_images
}

print("English training questions:", len(english_rows))
print("Unique images with English questions:", len(question_images))
print("Question images found in archive:", len(question_images & source_files))
print("Question image folders with detection.json:", len(question_folders & detection_folders))
print("Example question image paths:", sorted(question_images)[:5])

English training questions: 4919
Unique images with English questions: 450
Question images found in archive: 450
Question image folders with detection.json: 450
Example question image paths: ['xmlab1/source.jpg', 'xmlab100/source.jpg', 'xmlab101/source.jpg', 'xmlab104/source.jpg', 'xmlab107/source.jpg']


In [8]:
import json
import os
import re
from pathlib import PurePosixPath
from zipfile import ZipFile

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

training_records = []
grounded_count = 0
missing_images = []

with ZipFile(imgs_zip_path) as archive:
    archive_files = [
        name.replace("\\", "/")
        for name in archive.namelist()
    ]

    for row in english_rows:
        image_name = row["img_name"]
        folder = PurePosixPath(image_name).parent.as_posix()

        image_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + image_name)),
            None
        )
        detection_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + folder + "/detection.json")),
            None
        )

        if image_zip_path is None:
            missing_images.append(image_name)
            continue

        # Extract the image so the training dataset can load it by path.
        image_path = os.path.join("/content/slake_images", image_name)
        os.makedirs(os.path.dirname(image_path), exist_ok=True)

        if not os.path.exists(image_path):
            with open(image_path, "wb") as image_file:
                image_file.write(archive.read(image_zip_path))

        # Read this image's organ boxes.
        detections = []
        if detection_zip_path is not None:
            detections = json.loads(archive.read(detection_zip_path))

        question = row["question"].lower()
        answer = row["answer"].strip().lower()
        bbox = None
        matched_organ = None

        # Attach a box only when the question names that organ and
        # the answer indicates it is present or asks where it is.
        for item in detections:
            for organ, box in item.items():
                organ_words = re.findall(r"[a-z]+", organ.lower())
                question_words = re.findall(r"[a-z]+", question)

                if organ_words and all(word in question_words for word in organ_words):
                    asks_presence = any(
                        word in question
                        for word in ["contain", "present", "visible", "is there"]
                    )
                    asks_location = any(
                        word in question
                        for word in ["where", "which side", "position"]
                    )

                    positive_answer = answer in {
                        "yes", "right", "left", "present"
                    }

                    if positive_answer and (asks_presence or asks_location):
                        bbox = box
                        matched_organ = organ
                        break
            if bbox is not None:
                break

        if bbox is not None:
            grounded_count += 1

        training_records.append({
            "img_id": row["img_id"],
            "img_name": image_name,
            "image_path": image_path,
            "qid": row["qid"],
            "question": row["question"],
            "answer": row["answer"],
            "bbox_xywh_pixels": bbox,
            "grounded_organ": matched_organ,
        })

print("Training records created:", len(training_records))
print("Records with a matched box:", grounded_count)
print("Missing images:", len(missing_images))
print("Example record:", json.dumps(training_records[0], indent=2))

Training records created: 4919
Records with a matched box: 174
Missing images: 0
Example record: {
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "image_path": "/content/slake_images/xmlab1/source.jpg",
  "qid": 0,
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "bbox_xywh_pixels": null,
  "grounded_organ": null
}


In [9]:
from collections import Counter
from PIL import Image

grounded_records = [
    row for row in training_records
    if row["bbox_xywh_pixels"] is not None
]

print("Grounded examples:")
for row in grounded_records[:5]:
    print(
        f"QID {row['qid']}: {row['question']} "
        f"-> {row['answer']} | organ={row['grounded_organ']} "
        f"| box={row['bbox_xywh_pixels']}"
    )

valid_boxes = 0
invalid_boxes = []

for row in grounded_records:
    x, y, width, height = row["bbox_xywh_pixels"]

    with Image.open(row["image_path"]) as image:
        image_width, image_height = image.size

    is_valid = (
        x >= 0 and y >= 0 and width > 0 and height > 0
        and x + width <= image_width
        and y + height <= image_height
    )

    if is_valid:
        valid_boxes += 1
    else:
        invalid_boxes.append(row["img_name"])

print("\nGrounded records:", len(grounded_records))
print("Boxes inside image boundaries:", valid_boxes)
print("Invalid boxes:", len(invalid_boxes))
print("Grounded organ counts:", dict(Counter(
    row["grounded_organ"] for row in grounded_records
)))

Grounded examples:
QID 3: Does the picture contain liver? -> Yes | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 6: Where is the liver? -> Right | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 68: Does the picture contain liver? -> Yes | organ=Liver | box=[50.0, 102.0, 109.0, 151.0]
QID 70: Does the picture contain spleen? -> Yes | organ=Spleen | box=[190.0, 175.0, 75.0, 69.0]
QID 244: Does the picture contain liver? -> Yes | organ=Liver | box=[48.0, 141.0, 55.0, 106.0]

Grounded records: 174
Boxes inside image boundaries: 174
Invalid boxes: 0
Grounded organ counts: {'Liver': 119, 'Spleen': 20, 'Brain Stem': 1, 'Spinal Cord': 11, 'Esophagus': 2, 'Brain Edema': 1, 'Stomach': 3, 'Small Bowel': 7, 'Heart': 2, 'Bladder': 1, 'Colon': 7}


In [ ]:
from collections import Counter
from PIL import Image

grounded_records = [
    row for row in training_records
    if row["bbox_xywh_pixels"] is not None
]

print("Grounded examples:")
for row in grounded_records[:5]:
    print(
        f"QID {row['qid']}: {row['question']} "
        f"-> {row['answer']} | organ={row['grounded_organ']} "
        f"| box={row['bbox_xywh_pixels']}"
    )

valid_boxes = 0
invalid_boxes = []

for row in grounded_records:
    x, y, width, height = row["bbox_xywh_pixels"]

    with Image.open(row["image_path"]) as image:
        image_width, image_height = image.size

    is_valid = (
        x >= 0 and y >= 0 and width > 0 and height > 0
        and x + width <= image_width
        and y + height <= image_height
    )

    if is_valid:
        valid_boxes += 1
    else:
        invalid_boxes.append(row["img_name"])

print("\nGrounded records:", len(grounded_records))
print("Boxes inside image boundaries:", valid_boxes)
print("Invalid boxes:", len(invalid_boxes))
print("Grounded organ counts:", dict(Counter(
    row["grounded_organ"] for row in grounded_records
)))

In [1]:
import importlib.metadata as metadata
import torch

print("PyTorch:", torch.__version__)

for package in ["transformers", "peft", "bitsandbytes", "trl"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU memory (GB):", round(gpu.total_memory / 1024**3, 2))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

for name in ["model", "gemma_model", "processor", "gemma_processor"]:
    value = globals().get(name)
    if value is not None:
        print(f"{name} type:", type(value).__name__)

PyTorch: 2.11.0+cu130
transformers: 5.18.0
peft: 0.21.1
bitsandbytes: not installed
trl: not installed

CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56
BF16 supported: True


In [2]:
print("GPU compute capability:", torch.cuda.get_device_capability(0))
print("Native BF16 capability check:",
      torch.cuda.get_device_capability(0) >= (8, 0))

print("\nNotebook variables containing model, processor, or gemma:")
for name, value in list(globals().items()):
    if any(word in name.lower() for word in ["model", "processor", "gemma"]):
        if not name.startswith("_"):
            print(f"{name}: {type(value).__name__}")

GPU compute capability: (7, 5)
Native BF16 capability check: False

Notebook variables containing model, processor, or gemma:


In [3]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00:00:0100:01


In [4]:
import bitsandbytes as bnb
import torch

print("bitsandbytes version:", bnb.__version__)
print("4-bit layer available:", hasattr(bnb.nn, "Linear4bit"))
print("CUDA available:", torch.cuda.is_available())

bitsandbytes version: 0.50.2
4-bit layer available: True
CUDA available: True


In [5]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

print("Model class:", type(model).__name__)
print("Processor class:", type(processor).__name__)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("Device map:", getattr(model, "hf_device_map", "not available"))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model class: Gemma4ForConditionalGeneration
Processor class: Gemma4Processor
Loaded in 4-bit: True
Device map: not available
GPU memory used (GB): 6.28


In [6]:
from collections import Counter
from torch.nn import Linear
from bitsandbytes.nn import Linear4bit

linear_modules = [
    (name, type(module).__name__)
    for name, module in model.named_modules()
    if isinstance(module, (Linear, Linear4bit))
]

print("Model type:", model.config.model_type)
print("Linear layer count:", len(linear_modules))
print("Layer name endings:", dict(Counter(
    name.split(".")[-1] for name, _ in linear_modules
)))

print("\nFirst 30 linear layer names:")
for name, module_type in linear_modules[:30]:
    print(f"{name} [{module_type}]")

Model type: gemma4
Linear layer count: 526
Layer name endings: {'input_proj': 1, 'linear': 232, 'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35, 'gate_proj': 35, 'up_proj': 35, 'down_proj': 35, 'per_layer_input_gate': 35, 'per_layer_projection': 35, 'per_layer_model_projection': 1, 'input_proj_linear': 1, 'relative_k_proj': 12, 'output_proj': 1, 'embedding_projection': 2, 'lm_head': 1}

First 30 linear layer names:
model.vision_tower.patch_embedder.input_proj [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.q_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.k_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.v_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.self_attn.o_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.gate_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.up_proj.linear [Linear4bit]
model.vision_tower.encoder.layers.0.mlp.down_proj.linear [Linear4bit]
model.vision_tow

In [7]:
for name, module in model.named_modules():
    if "language_model" in name and (
        name.endswith(".q_proj") or name.endswith(".q_proj.linear")
    ):
        print(name, "->", type(module).__name__)

model.language_model.layers.0.self_attn.q_proj -> Linear4bit
model.language_model.layers.1.self_attn.q_proj -> Linear4bit
model.language_model.layers.2.self_attn.q_proj -> Linear4bit
model.language_model.layers.3.self_attn.q_proj -> Linear4bit
model.language_model.layers.4.self_attn.q_proj -> Linear4bit
model.language_model.layers.5.self_attn.q_proj -> Linear4bit
model.language_model.layers.6.self_attn.q_proj -> Linear4bit
model.language_model.layers.7.self_attn.q_proj -> Linear4bit
model.language_model.layers.8.self_attn.q_proj -> Linear4bit
model.language_model.layers.9.self_attn.q_proj -> Linear4bit
model.language_model.layers.10.self_attn.q_proj -> Linear4bit
model.language_model.layers.11.self_attn.q_proj -> Linear4bit
model.language_model.layers.12.self_attn.q_proj -> Linear4bit
model.language_model.layers.13.self_attn.q_proj -> Linear4bit
model.language_model.layers.14.self_attn.q_proj -> Linear4bit
model.language_model.layers.15.self_attn.q_proj -> Linear4bit
model.language_mod

In [8]:
from collections import Counter
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}

lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and ".self_attn." in name
    and name.rsplit(".", 1)[-1] in target_suffixes
]

print("LoRA target count:", len(lora_targets))
print("Targets by type:", dict(Counter(
    name.rsplit(".", 1)[-1] for name in lora_targets
)))

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

LoRA target count: 100
Targets by type: {'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35}


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.75 GiB. GPU 0 has a total capacity of 14.56 GiB of which 6.58 GiB is free. Including non-PyTorch memory, this process has 7.98 GiB memory in use. Of the allocated memory 7.06 GiB is allocated by PyTorch, and 825.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [9]:
import gc
import torch
from transformers import AutoModelForMultimodalLM

# Remove the model that may have been partly changed by the failed call.
del model
gc.collect()
torch.cuda.empty_cache()

# Reload the clean 4-bit model.
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

# Freeze the base model without converting its weights to FP32.
for parameter in model.parameters():
    parameter.requires_grad = False

model.config.use_cache = False

print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("Trainable base parameters:",
      sum(parameter.numel() for parameter in model.parameters()
          if parameter.requires_grad))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [10]:
import torch

free_bytes, total_bytes = torch.cuda.mem_get_info()

print("GPU total (GB):", round(total_bytes / 1024**3, 2))
print("GPU currently free (GB):", round(free_bytes / 1024**3, 2))
print("PyTorch allocated (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))
print("PyTorch reserved (GB):", round(
    torch.cuda.memory_reserved() / 1024**3, 2
))

GPU total (GB): 14.56
GPU currently free (GB): 6.58
PyTorch allocated (GB): 7.06
PyTorch reserved (GB): 7.87


In [11]:
import json
import os

checkpoint_path = "/content/slake_training_records.json"

with open(checkpoint_path, "w", encoding="utf-8") as file:
    json.dump(training_records, file)

print("Saved records:", len(training_records))
print("Checkpoint:", checkpoint_path)
print("File size (MB):", round(
    os.path.getsize(checkpoint_path) / 1024**2, 2
))

NameError: name 'training_records' is not defined

In [12]:
for name in [
    "training_records",
    "slake_train_data",
    "imgs_zip_path",
    "model",
    "processor",
    "torch",
]:
    print(f"{name}:", "available" if name in globals() else "missing")

if "torch" in globals() and torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))
    print("PyTorch allocated (GB):", round(
        torch.cuda.memory_allocated() / 1024**3, 2
    ))

training_records: missing
slake_train_data: missing
imgs_zip_path: missing
model: missing
processor: available
torch: available
GPU free (GB): 6.58
PyTorch allocated (GB): 7.06


In [2]:
import torch

free_bytes, total_bytes = torch.cuda.mem_get_info()
print("GPU total (GB):", round(total_bytes / 1024**3, 2))
print("GPU free (GB):", round(free_bytes / 1024**3, 2))
print("PyTorch allocated (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

AssertionError: Torch not compiled with CUDA enabled

In [3]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Python: /usr/bin/python3
PyTorch: 2.11.0+cpu
CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU free (GB): 14.46


In [2]:
import importlib.metadata as metadata
import os
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("\nPackage versions:")

for package in ["transformers", "peft", "bitsandbytes", "datasets"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nFiles currently in /content:")
print(os.listdir("/content")[:20])

GPU: Tesla T4

Package versions:
transformers: 5.18.0
peft: 0.21.1
bitsandbytes: not installed
datasets: 4.8.5

Files currently in /content:
['.config', 'sample_data']


In [3]:
%pip install -q bitsandbytes==0.50.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.9 MB/s eta 0:00:00:00:0100:01


In [4]:
import torch
import bitsandbytes
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

print("Model:", type(model).__name__)
print("Processor:", type(processor).__name__)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Processor: Gemma4Processor
Loaded in 4-bit: True
GPU memory used (GB): 6.28


In [5]:
from collections import Counter
from peft import LoraConfig, TaskType, get_peft_model

# Freeze the quantized base model without changing its weight precision.
for parameter in model.parameters():
    parameter.requires_grad = False

model.config.use_cache = False

# Target attention projections in the language model only.
target_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj"}
lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and ".self_attn." in name
    and name.rsplit(".", 1)[-1] in target_suffixes
]

print("LoRA target count:", len(lora_targets))
print("Targets by type:", dict(Counter(
    name.rsplit(".", 1)[-1] for name in lora_targets
)))

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("GPU memory used (GB):", round(
    torch.cuda.memory_allocated() / 1024**3, 2
))

LoRA target count: 100
Targets by type: {'q_proj': 35, 'k_proj': 15, 'v_proj': 15, 'o_proj': 35}
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525
GPU memory used (GB): 6.29


In [1]:
import json
import os
from huggingface_hub import hf_hub_download

train_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="train.json",
    repo_type="dataset",
)

imgs_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset",
)

with open(train_path, "r", encoding="utf-8") as file:
    slake_train_data = json.load(file)

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

print("Training examples:", len(slake_train_data))
print("English training questions:", len(english_rows))
print("Images archive found:", os.path.exists(imgs_zip_path))
print("Images archive size (MB):", round(
    os.path.getsize(imgs_zip_path) / 1024**2, 2
))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            

Training examples: 9835
English training questions: 4919
Images archive found: True
Images archive size (MB): 202.51


In [2]:
import json
import os
import re
from pathlib import PurePosixPath
from zipfile import ZipFile

training_records = []
missing_images = []

with ZipFile(imgs_zip_path) as archive:
    archive_files = [
        name.replace("\\", "/")
        for name in archive.namelist()
    ]

    for row in english_rows:
        image_name = row["img_name"]
        folder = PurePosixPath(image_name).parent.as_posix()

        image_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + image_name)),
            None
        )
        detection_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + folder + "/detection.json")),
            None
        )

        if image_zip_path is None:
            missing_images.append(image_name)
            continue

        image_path = os.path.join("/content/slake_images", image_name)
        os.makedirs(os.path.dirname(image_path), exist_ok=True)

        if not os.path.exists(image_path):
            with open(image_path, "wb") as image_file:
                image_file.write(archive.read(image_zip_path))

        detections = []
        if detection_zip_path is not None:
            detections = json.loads(archive.read(detection_zip_path))

        question = row["question"].lower()
        answer = row["answer"].strip().lower()
        question_words = re.findall(r"[a-z]+", question)

        bbox = None
        grounded_organ = None

        for item in detections:
            for organ, box in item.items():
                organ_words = re.findall(r"[a-z]+", organ.lower())

                if not organ_words or not all(
                    word in question_words for word in organ_words
                ):
                    continue

                asks_presence = any(
                    word in question
                    for word in ["contain", "present", "visible", "is there"]
                )
                asks_location = any(
                    word in question
                    for word in ["where", "which side", "position"]
                )

                if answer in {"yes", "right", "left", "present"} and (
                    asks_presence or asks_location
                ):
                    bbox = box
                    grounded_organ = organ
                    break
            if bbox is not None:
                break

        training_records.append({
            "img_id": row["img_id"],
            "img_name": image_name,
            "image_path": image_path,
            "qid": row["qid"],
            "question": row["question"],
            "answer": row["answer"],
            "bbox_xywh_pixels": bbox,
            "grounded_organ": grounded_organ,
        })

grounded_count = sum(
    row["bbox_xywh_pixels"] is not None
    for row in training_records
)

print("Training records:", len(training_records))
print("Matched boxes:", grounded_count)
print("Missing images:", len(missing_images))
print("Example:", training_records[0])

Training records: 4919
Matched boxes: 174
Missing images: 0
Example: {'img_id': 1, 'img_name': 'xmlab1/source.jpg', 'image_path': '/content/slake_images/xmlab1/source.jpg', 'qid': 0, 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'bbox_xywh_pixels': None, 'grounded_organ': None}


In [3]:
import json
from huggingface_hub import hf_hub_download

validation_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="validation.json",
    repo_type="dataset",
)

with open(validation_path, "r", encoding="utf-8") as file:
    slake_validation_data = json.load(file)

validation_english_rows = [
    row for row in slake_validation_data
    if row.get("q_lang") == "en"
]

validation_images = {
    row["img_name"] for row in validation_english_rows
}

print("Validation examples:", len(slake_validation_data))
print("English validation questions:", len(validation_english_rows))
print("Unique validation images:", len(validation_images))
print("Example:", validation_english_rows[0])

validation.json:   0%|          | 0.00/639k [00:00<?, ?B/s]

Validation examples: 2099
English validation questions: 1053
Unique validation images: 96
Example: {'img_id': 0, 'img_name': 'xmlab0/source.jpg', 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'q_lang': 'en', 'location': 'Abdomen', 'modality': 'MRI', 'answer_type': 'OPEN', 'base_type': 'vqa', 'content_type': 'Modality', 'triple': ['vhead', '_', '_'], 'qid': 9835}


In [4]:
import json
import os
import re
from pathlib import PurePosixPath
from zipfile import ZipFile

validation_records = []
missing_validation_images = []

with ZipFile(imgs_zip_path) as archive:
    archive_files = [
        name.replace("\\", "/")
        for name in archive.namelist()
    ]

    for row in validation_english_rows:
        image_name = row["img_name"]
        folder = PurePosixPath(image_name).parent.as_posix()

        image_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + image_name)),
            None
        )
        detection_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + folder + "/detection.json")),
            None
        )

        if image_zip_path is None:
            missing_validation_images.append(image_name)
            continue

        image_path = os.path.join("/content/slake_images", image_name)
        os.makedirs(os.path.dirname(image_path), exist_ok=True)

        if not os.path.exists(image_path):
            with open(image_path, "wb") as image_file:
                image_file.write(archive.read(image_zip_path))

        detections = []
        if detection_zip_path is not None:
            detections = json.loads(archive.read(detection_zip_path))

        question = row["question"].lower()
        answer = row["answer"].strip().lower()
        question_words = re.findall(r"[a-z]+", question)

        bbox = None
        grounded_organ = None

        for item in detections:
            for organ, box in item.items():
                organ_words = re.findall(r"[a-z]+", organ.lower())

                if not organ_words or not all(
                    word in question_words for word in organ_words
                ):
                    continue

                asks_presence = any(
                    word in question
                    for word in ["contain", "present", "visible", "is there"]
                )
                asks_location = any(
                    word in question
                    for word in ["where", "which side", "position"]
                )

                if answer in {"yes", "right", "left", "present"} and (
                    asks_presence or asks_location
                ):
                    bbox = box
                    grounded_organ = organ
                    break
            if bbox is not None:
                break

        validation_records.append({
            "img_id": row["img_id"],
            "img_name": image_name,
            "image_path": image_path,
            "qid": row["qid"],
            "question": row["question"],
            "answer": row["answer"],
            "bbox_xywh_pixels": bbox,
            "grounded_organ": grounded_organ,
        })

print("Validation records:", len(validation_records))
print("Matched boxes:", sum(
    row["bbox_xywh_pixels"] is not None
    for row in validation_records
))
print("Missing images:", len(missing_validation_images))
print("Example:", validation_records[0])

Validation records: 1053
Matched boxes: 31
Missing images: 0
Example: {'img_id': 0, 'img_name': 'xmlab0/source.jpg', 'image_path': '/content/slake_images/xmlab0/source.jpg', 'qid': 9835, 'question': 'What modality is used to take this image?', 'answer': 'MRI', 'bbox_xywh_pixels': None, 'grounded_organ': None}


In [5]:
import json
from PIL import Image

image_sizes = {}

def add_target_text(row):
    image_path = row["image_path"]

    if image_path not in image_sizes:
        with Image.open(image_path) as image:
            image_sizes[image_path] = image.size

    image_width, image_height = image_sizes[image_path]
    bbox = row["bbox_xywh_pixels"]

    if bbox is None:
        region = None
    else:
        x, y, width, height = bbox
        region = {
            "organ": row["grounded_organ"],
            "bbox_xyxy_normalized": [
                round(x / image_width, 4),
                round(y / image_height, 4),
                round((x + width) / image_width, 4),
                round((y + height) / image_height, 4),
            ],
        }

    example = row.copy()
    example["target_text"] = json.dumps(
        {"answer": row["answer"], "region": region},
        separators=(",", ":"),
    )
    return example

training_examples = [
    add_target_text(row) for row in training_records
]
validation_examples = [
    add_target_text(row) for row in validation_records
]

print("Example without a box:")
print(training_examples[0]["target_text"])

grounded_example = next(
    row for row in training_examples
    if row["bbox_xywh_pixels"] is not None
)
print("\nExample with a box:")
print(grounded_example["question"])
print(grounded_example["target_text"])

Example without a box:
{"answer":"MRI","region":null}

Example with a box:
Does the picture contain liver?
{"answer":"Yes","region":{"organ":"Liver","bbox_xyxy_normalized":[0.2109,0.4141,0.3281,0.5352]}}


In [6]:
sample = training_examples[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": sample["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {sample['question']}"
                ),
            },
        ],
    },
    {
        "role": "assistant",
        "content": sample["target_text"],
    },
]

encoded = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=False,
    return_assistant_tokens_mask=True,
)

print("Question:", sample["question"])
print("Target:", sample["target_text"])
print("Processor output keys:", list(encoded.keys()))

for key, value in encoded.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Assistant mask available:", "assistant_masks" in encoded)
if "assistant_masks" in encoded:
    print("Assistant tokens marked:", int(encoded["assistant_masks"].sum()))

NameError: name 'processor' is not defined

In [7]:
from transformers import AutoProcessor

MODEL_ID = "google/gemma-4-E2B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Processor:", type(processor).__name__)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

Processor: Gemma4Processor


In [8]:
sample = training_examples[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": sample["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {sample['question']}"
                ),
            },
        ],
    },
    {
        "role": "assistant",
        "content": sample["target_text"],
    },
]

encoded = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=False,
    return_assistant_tokens_mask=True,
)

print("Question:", sample["question"])
print("Target:", sample["target_text"])
print("Processor output keys:", list(encoded.keys()))

for key, value in encoded.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Assistant mask available:", "assistant_masks" in encoded)
if "assistant_masks" in encoded:
    print("Assistant tokens marked:", int(encoded["assistant_masks"].sum()))

[transformers] return_assistant_tokens_mask==True but chat template does not contain `{% generation %}` keyword.


Question: What modality is used to take this image?
Target: {"answer":"MRI","region":null}
Processor output keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'assistant_masks']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
assistant_masks shape: (1, 301)
Assistant mask available: True
Assistant tokens marked: 0


In [9]:
prompt_encoded = processor.apply_chat_template(
    messages[:1],  # user message only
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
)

prompt_ids = prompt_encoded["input_ids"][0]
full_ids = encoded["input_ids"][0]

same_prefix = (
    len(full_ids) >= len(prompt_ids)
    and (full_ids[:len(prompt_ids)] == prompt_ids).all().item()
)

print("Prompt token count:", len(prompt_ids))
print("Full example token count:", len(full_ids))
print("Prompt matches start of full example:", same_prefix)

Prompt token count: 290
Full example token count: 301
Prompt matches start of full example: True


In [10]:
labels = encoded["input_ids"].clone()
labels[:, :len(prompt_ids)] = -100

supervised_ids = labels[0][labels[0] != -100]

print("Prompt tokens masked:", int((labels == -100).sum()))
print("Answer tokens used for training:", len(supervised_ids))
print(
    "Text used for training:",
    processor.tokenizer.decode(supervised_ids, skip_special_tokens=False),
)

Prompt tokens masked: 290
Answer tokens used for training: 11
Text used for training: {"answer":"MRI","region":null}<turn|>



In [11]:
def collate_vqa(batch):
    if len(batch) != 1:
        raise ValueError("This collator currently expects batch size 1.")

    example = batch[0]

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "url": example["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {example['question']}"
                ),
            },
        ],
    }

    full_messages = [
        user_message,
        {"role": "assistant", "content": example["target_text"]},
    ]

    full = processor.apply_chat_template(
        full_messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=False,
    )

    prompt = processor.apply_chat_template(
        [user_message],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt tokens do not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100
    full["labels"] = labels

    return dict(full)


test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print("Answer tokens used for training:",
      int((test_batch["labels"] != -100).sum()))

NameError: name 'torch' is not defined

In [12]:
import torch

test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))

print(
    "Answer tokens used for training:",
    int((test_batch["labels"] != -100).sum()),
)

Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
labels shape: (1, 301)
Answer tokens used for training: 11


In [13]:
from transformers import Trainer, TrainingArguments

print("Training examples:", len(training_examples))
print("Validation examples:", len(validation_examples))
print("Model:", type(model).__name__)

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    eval_dataset=validation_examples,
    data_collator=collate_vqa,
    processing_class=processor,
)

print("Trainer created successfully.")
print("Training has not started.")

Training examples: 4919
Validation examples: 1053


NameError: name 'model' is not defined

In [14]:
print("Model-like variables currently loaded:")

for name, value in list(globals().items()):
    if ("model" in name.lower() or "gemma" in name.lower()):
        if hasattr(value, "parameters"):
            print(name, "->", type(value).__name__)

print("LoRA config available:", "lora_config" in globals())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU free (GB):", round(free_bytes / 1024**3, 2))

Model-like variables currently loaded:
LoRA config available: False
CUDA available: True
GPU free (GB): 14.46


In [15]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [16]:
import importlib.util

print("bitsandbytes package found:",
      importlib.util.find_spec("bitsandbytes") is not None)

try:
    import bitsandbytes as bnb
    print("bitsandbytes version:", bnb.__version__)
except Exception as error:
    print("Import failed:", type(error).__name__)
    print("Details:", str(error))

bitsandbytes package found: False
Import failed: ModuleNotFoundError
Details: No module named 'bitsandbytes'


In [17]:
%pip install -q bitsandbytes==0.50.2

import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00:00:0100:01
bitsandbytes version: 0.50.2
CUDA available: True


In [18]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [19]:
import transformers
from transformers.utils import is_bitsandbytes_available, is_accelerate_available

print("Transformers:", transformers.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())
print("Accelerate available:", is_accelerate_available())

Transformers: 5.18.0
Transformers detects bitsandbytes: False
Accelerate available: True


In [20]:
import json

with open("/content/training_examples_step7_backup.json", "w", encoding="utf-8") as f:
    json.dump(training_examples, f, ensure_ascii=False)

with open("/content/validation_examples_step7_backup.json", "w", encoding="utf-8") as f:
    json.dump(validation_examples, f, ensure_ascii=False)

print("Training examples saved:", len(training_examples))
print("Validation examples saved:", len(validation_examples))
print("Backup files saved in /content")

Training examples saved: 4919
Validation examples saved: 1053
Backup files saved in /content


In [21]:
import json
import torch
import bitsandbytes as bnb
import transformers
from transformers.utils import is_bitsandbytes_available

with open("/content/training_examples_step7_backup.json", encoding="utf-8") as f:
    training_examples = json.load(f)

with open("/content/validation_examples_step7_backup.json", encoding="utf-8") as f:
    validation_examples = json.load(f)

print("Training examples restored:", len(training_examples))
print("Validation examples restored:", len(validation_examples))
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("Transformers detects bitsandbytes:", is_bitsandbytes_available())
print("CUDA available:", torch.cuda.is_available())

Training examples restored: 4919
Validation examples restored: 1053
Transformers: 5.18.0
bitsandbytes: 0.50.2
Transformers detects bitsandbytes: False
CUDA available: True


In [22]:
import importlib.util
import importlib.metadata
import transformers.utils.import_utils as import_utils

spec = importlib.util.find_spec("bitsandbytes")

print("bitsandbytes module path:", spec.origin if spec else "not found")

try:
    print("bitsandbytes package version:",
          importlib.metadata.version("bitsandbytes"))
except Exception as error:
    print("Package metadata error:", type(error).__name__, str(error))

print("Transformers cached bitsandbytes flag:",
      getattr(import_utils, "_bitsandbytes_available", "not found"))
print("Transformers bitsandbytes check:",
      import_utils.is_bitsandbytes_available())

bitsandbytes module path: /usr/local/lib/python3.13/dist-packages/bitsandbytes/__init__.py
bitsandbytes package version: 0.50.2
Transformers cached bitsandbytes flag: not found
Transformers bitsandbytes check: False


In [23]:
import inspect
from transformers.utils import import_utils

print(inspect.getsource(import_utils.is_bitsandbytes_available))

@lru_cache
@_make_compile_constant
def is_bitsandbytes_available(min_version: str = BITSANDBYTES_MIN_VERSION) -> bool:
    is_available, bitsandbytes_version = _is_package_available("bitsandbytes", return_version=True)
    return is_available and version.parse(bitsandbytes_version) >= version.parse(min_version)



In [24]:
import transformers.utils.import_utils as import_utils

print("Required bitsandbytes version:",
      import_utils.BITSANDBYTES_MIN_VERSION)
print("Package detection:",
      import_utils._is_package_available("bitsandbytes", return_version=True))

import_utils.is_bitsandbytes_available.cache_clear()

print("Detection after clearing cache:",
      import_utils.is_bitsandbytes_available())

Required bitsandbytes version: 0.46.1
Package detection: (True, '0.50.2')
Detection after clearing cache: True


In [25]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Model:", type(model).__name__)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("GPU memory used (GB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model: Gemma4ForConditionalGeneration
Loaded in 4-bit: True
GPU memory used (GB): 6.28


In [26]:
from peft import LoraConfig, get_peft_model

target_suffixes = ("q_proj", "k_proj", "v_proj", "o_proj")

lora_targets = [
    name
    for name, module in model.named_modules()
    if name.startswith("model.language_model.layers.")
    and name.endswith(target_suffixes)
    and module.__class__.__name__ == "Linear4bit"
]

print("LoRA target count:", len(lora_targets))

for parameter in model.parameters():
    parameter.requires_grad = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=lora_targets,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

LoRA target count: 100
trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525


In [27]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Processor:", type(processor).__name__)
print("Training examples:", len(training_examples))
print("Validation examples:", len(validation_examples))

Processor: Gemma4Processor
Training examples: 4919
Validation examples: 1053


In [28]:
import torch

def collate_vqa(batch):
    if len(batch) != 1:
        raise ValueError("This collator expects batch size 1.")

    example = batch[0]

    user_message = {
        "role": "user",
        "content": [
            {"type": "image", "url": example["image_path"]},
            {
                "type": "text",
                "text": (
                    "Answer the radiology question. Return only the requested JSON.\n"
                    f"Question: {example['question']}"
                ),
            },
        ],
    }

    full = processor.apply_chat_template(
        [
            user_message,
            {"role": "assistant", "content": example["target_text"]},
        ],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=False,
    )

    prompt = processor.apply_chat_template(
        [user_message],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )

    prompt_ids = prompt["input_ids"][0]
    full_ids = full["input_ids"][0]

    if not torch.equal(full_ids[:len(prompt_ids)], prompt_ids):
        raise ValueError("Prompt does not match the start of the full example.")

    labels = full["input_ids"].clone()
    labels[:, :len(prompt_ids)] = -100
    full["labels"] = labels

    return dict(full)


test_batch = collate_vqa([training_examples[0]])

print("Batch keys:", list(test_batch.keys()))
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key} shape:", tuple(value.shape))
print("Answer tokens used for training:",
      int((test_batch["labels"] != -100).sum()))

Batch keys: ['input_ids', 'attention_mask', 'mm_token_type_ids', 'pixel_values', 'image_position_ids', 'labels']
input_ids shape: (1, 301)
attention_mask shape: (1, 301)
mm_token_type_ids shape: (1, 301)
pixel_values shape: (1, 2520, 768)
image_position_ids shape: (1, 2520, 2)
labels shape: (1, 301)
Answer tokens used for training: 11


In [29]:
from transformers import Trainer, TrainingArguments

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/gemma4_slake_checkpoints",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_examples,
    eval_dataset=validation_examples,
    data_collator=collate_vqa,
    processing_class=processor,
)

print("Trainer created successfully.")
print("Training has not started.")

Trainer created successfully.
Training has not started.
